# OpenRA-RL: Teaching an LLM to Play Red Alert 🎮

This notebook demonstrates the full pipeline:
1. **Connect** to the OpenRA-RL environment
2. **Collect** expert demonstrations from the ScriptedBot
3. **Train** a small LLM via behavioral cloning (imitation learning)
4. **Evaluate** the trained model

## Prerequisites
```bash
# Start the OpenRA-RL server:
docker run -p 8000:8000 openra-rl

# Install dependencies:
pip install -r requirements.txt
```

## 1. Explore the Environment

Let's connect to OpenRA-RL and see what a game observation looks like.

In [ ]:
import sys
sys.path.insert(0, '../')
sys.path.insert(0, '../../OpenRA-RL')
sys.path.insert(0, '../../OpenRA-RL/examples')

import asyncio
import json
from openra_env.client import OpenRAEnv
from openra_env.models import OpenRAAction, CommandModel, ActionType

ENV_URL = "http://localhost:8000"

In [ ]:
async def explore_env():
    """Connect, reset, and inspect the initial observation."""
    async with OpenRAEnv(base_url=ENV_URL, message_timeout_s=300.0) as env:
        result = await env.reset()
        obs = result.observation
        
        print(f"Map: {obs.map_info.map_name} ({obs.map_info.width}x{obs.map_info.height})")
        print(f"Tick: {obs.tick}")
        print(f"\nEconomy:")
        print(f"  Cash: ${obs.economy.cash}")
        print(f"  Power: {obs.economy.power_provided}/{obs.economy.power_drained}")
        print(f"  Harvesters: {obs.economy.harvester_count}")
        
        print(f"\nUnits ({len(obs.units)}):")
        for u in obs.units[:5]:
            print(f"  {u.type} #{u.actor_id} at ({u.cell_x}, {u.cell_y}) {'IDLE' if u.is_idle else u.current_activity}")
        
        print(f"\nBuildings ({len(obs.buildings)}):")
        for b in obs.buildings[:5]:
            print(f"  {b.type} #{b.actor_id} at ({b.cell_x}, {b.cell_y})")
            
        print(f"\nAvailable to build: {', '.join(obs.available_production[:10])}")
        return obs

obs = await explore_env()

## 2. Watch the ScriptedBot Play

The ScriptedBot is our "expert". It follows a fixed strategy:
1. Deploy MCV → build base (power, barracks, refinery, war factory)
2. Train infantry + APC, set stances, assign guards
3. Attack-move toward enemy with full army

In [ ]:
from scripted_bot import ScriptedBot

async def watch_bot(max_steps=500):
    """Run the ScriptedBot for a few steps and show what it does."""
    bot = ScriptedBot(verbose=True)
    
    async with OpenRAEnv(base_url=ENV_URL, message_timeout_s=300.0) as env:
        result = await env.reset()
        print(f"Game started! Map: {result.observation.map_info.map_name}")
        
        step = 0
        while not result.done and step < max_steps:
            action = bot.decide(result.observation)
            result = await env.step(action)
            step += 1
            
            if step % 100 == 0:
                obs = result.observation
                print(f"\nStep {step} | Tick {obs.tick} | ${obs.economy.cash} | "
                      f"Units:{len(obs.units)} | Bldgs:{len(obs.buildings)} | {bot.phase}")
        
        obs = result.observation
        print(f"\nFinal: {obs.result or 'timeout'} | {step} steps | "
              f"Killed {obs.military.units_killed}u | Lost {obs.military.units_lost}u")

await watch_bot(max_steps=500)

## 3. Collect Expert Data

Now let's collect full game trajectories for training.

In [ ]:
from scripts.collect_bot_data import collect_episode, serialize_obs, serialize_action
from pathlib import Path
import time

NUM_EPISODES = 5
OUTPUT_DIR = Path("../data/episodes")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for i in range(1, NUM_EPISODES + 1):
    print(f"\nCollecting episode {i}/{NUM_EPISODES}...")
    t0 = time.time()
    
    trajectory = await collect_episode(ENV_URL, episode_id=i, max_steps=3000, verbose=True)
    
    filename = OUTPUT_DIR / f"episode_{i:03d}.json"
    with open(filename, "w") as f:
        json.dump(trajectory, f)
    
    elapsed = time.time() - t0
    final = trajectory[-1]["observation"]
    print(f"  Saved {filename.name} | {len(trajectory)} steps | "
          f"{final.get('result', 'timeout')} | {elapsed:.0f}s")

## 4. Prepare Training Data

Convert collected trajectories into prompt-completion pairs for SFT.

In [ ]:
from scripts.train_imitation import obs_to_prompt, action_to_completion, load_episodes

# Load and convert episodes
samples = load_episodes(OUTPUT_DIR)

print(f"\nTotal training samples: {len(samples)}")
print("\n" + "="*60)
print("EXAMPLE PROMPT:")
print("="*60)
print(samples[0]["prompt"])
print("\n" + "="*60)
print("EXAMPLE COMPLETION:")
print("="*60)
print(samples[0]["completion"])

## 5. Train (Behavioral Cloning)

Fine-tune a small LLM on the expert demonstrations.

⚠️ **Requires a GPU.** For a quick demo, use `--epochs 1` and a small model.

In [ ]:
# Option A: Train from the notebook
# (uncomment and run if you have a GPU)

# !python ../scripts/train_imitation.py \
#     --data-dir ../data/episodes \
#     --model Qwen/Qwen3-4B \
#     --epochs 1 \
#     --batch-size 2 \
#     --output-dir ../checkpoints/openra-il

print("To train, uncomment the command above or run:")
print("  python scripts/train_imitation.py --data-dir data/episodes --model Qwen/Qwen3-4B")

## 6. Evaluate with Shaped Reward

Score the collected episodes using the multi-dimensional evaluation reward.

In [ ]:
from rewards.shaped_reward import EvalReward

reward_fn = EvalReward()

# Load all episodes and score them
episode_files = sorted(OUTPUT_DIR.glob("episode_*.json"))
episodes = []
for ep_file in episode_files:
    with open(ep_file) as f:
        episodes.append(json.load(f))

results = reward_fn.compare(episodes)

# Print results table
print(f"{'Ep':>4} {'Explore':>8} {'Base':>8} {'Army':>8} {'Combat':>8} {'Surv':>8} {'Outcome':>8} {'TOTAL':>8}")
print("-" * 68)

for r in results["episodes"]:
    c = r["components"]
    print(f"{r['episode']:>4} {c['exploration']:>8.3f} {c['base_progress']:>8.3f} "
          f"{c['army_strength']:>8.3f} {c['combat_ratio']:>8.3f} "
          f"{c['survival']:>8.3f} {c['outcome']:>8.3f} {r['total']:>8.4f}")

print("-" * 68)
avg = results["averages"]
print(f"{'AVG':>4} {avg['exploration']:>8.3f} {avg['base_progress']:>8.3f} "
      f"{avg['army_strength']:>8.3f} {avg['combat_ratio']:>8.3f} "
      f"{avg['survival']:>8.3f} {avg['outcome']:>8.3f} {results['average_total']:>8.4f}")

## Summary

This notebook demonstrated:
- **OpenRA-RL** as an OpenEnv-compatible RTS game environment
- **Data collection** from an expert ScriptedBot playing full Red Alert games
- **Imitation learning** setup with prompt-completion formatting for LLM SFT
- **Multi-dimensional evaluation** using a shaped reward function

### Links
- [OpenRA-RL Repository](https://github.com/your-org/OpenRA-RL)
- [OpenEnv Framework](https://meta-pytorch.org/OpenEnv/)
- [HuggingFace Space](https://huggingface.co/openenv)